# GradCAM (Incomplete)
Create a GradCAM notebook that loads a Keras model from the "along-capstone-data" GCS bucket, specifically from the "models" folder, and uses it to generate GradCAM visualizations for images in the "along-capstone-data" GCS bucket, specifically from the "data" folder. The notebook should include steps for loading the model and data, preprocessing the images, generating predictions, calculating GradCAM heatmaps, and visualizing the results. The notebook should also include error handling for model and data loading, and for building the dataset index. The notebook should also include timing for the data loading operation.

### Insights or Next Steps

*   The primary blocker for generating Grad-CAM visualizations is the inability to successfully call the loaded Keras model within the gradient computation context. Further investigation is needed to understand why the model's `call()` method fails with the provided input tensor shape and structure in this specific setup, potentially involving examining the model's internal structure or Keras 3 compatibility issues.
*   If the model calling issue cannot be resolved, alternative approaches for Grad-CAM or other interpretability methods compatible with the loaded model structure might need to be explored.

## Setup

Install necessary libraries.


In [44]:
from pathlib import Path

# ===== USER TUNABLES =====
# MODEL_PATH = Path("models/baseline_savedmodel/resnet50_profilepic_classifier.keras")  # dir or file (.keras / .h5 / SavedModel dir)
MODEL_PATH = Path("models/resnet50_profilepic_classifier.keras")
CLASS_NAMES = None

DATA_ROOT = Path("final")
METADATA_CSV = None

TARGET_SPLIT = "val"
IMG_SIZE = 224
BATCH = 32

PREPROCESS = "resnet50"  # 'resnet50' | 'efficientnet' | 'none'
TARGET_LAYER_NAME = None # e.g., "conv5_block3_out"; None => auto-detect last conv
ALPHA = 0.35

N_MISCLASS_PER_CLASS = 8
N_CORRECT_PER_CLASS = 6

OUT_DIR = Path("gradcam_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# =========================

## Load model and metadata

Load the Keras model and associated metadata (like class names, image size, and the target convolutional layer name) from the specified paths.


In [45]:
# === GradcamInspection: DROP-IN REPLACEMENT for model + metadata loading ===
from pathlib import Path
from keras.models import load_model
import tensorflow as tf
import json

def load_model_and_meta(model_path: str | Path):
    p = Path(model_path)
    if not p.exists():
        raise FileNotFoundError(f"Model file not found: {p.resolve()}")

    # Load the Keras single-file model (no compile needed for inference)
    model = load_model(p, compile=False)

    # Sidecar paths
    classes_path = p.with_suffix(".classes.json")
    meta_path    = p.with_suffix(".meta.json")

    # --- Classes: try sidecar, else infer count and synthesize names ---
    if classes_path.exists():
        try:
            CLASS_NAMES_local = json.loads(classes_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {classes_path.name}: {e}")
            CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]
    else:
        CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]

    # --- Meta: img_size, last_conv, etc. (optional) ---
    meta_local = {}
    if meta_path.exists():
        try:
            meta_local = json.loads(meta_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {meta_path.name}: {e}")

    # Input info
    inp      = model.inputs[0]
    shape    = inp.shape
    INPUT_NAME_local  = inp.name.split(":")[0]
    INPUT_DTYPE_local = inp.dtype

    # IMG_SIZE: prefer sidecar; else infer from input tensor
    IMG_SIZE_local = int(meta_local.get("img_size") or (shape[1] if shape[1] is not None else 224))

    # Find a valid 4D feature map layer if not provided
    def _find_last_4d(m):
        for lyr in reversed(m.layers):
            try:
                s = getattr(lyr, "output_shape", None)
                if s is None:
                    continue
                if isinstance(s, (list, tuple)) and s and isinstance(s[0], tuple):
                    s = s[0]
                if hasattr(s, "__len__") and len(s) == 4:
                    return lyr.name
            except Exception:
                pass
        raise ValueError("No 4D feature map layer found; cannot run Grad-CAM.")
    LAST_CONV_local = meta_local.get("last_conv") or _find_last_4d(model)

    # Diagnostics
    print(f"Loaded: {p.name}")
    print(f"Input -> name='{INPUT_NAME_local}', dtype={INPUT_DTYPE_local.name}, shape={tuple(shape)}")
    print(f"IMG_SIZE={IMG_SIZE_local}  |  LAST_CONV='{LAST_CONV_local}'  |  #classes={len(CLASS_NAMES_local)}")

    # NOTE: Preprocessing is assumed to be embedded in the model graph.
    # Do NOT apply external preprocess_input in this notebook.

    return model, CLASS_NAMES_local, IMG_SIZE_local, INPUT_NAME_local, INPUT_DTYPE_local, LAST_CONV_local

# ---- call it ----
MODEL_PATH = "models/resnet50_profilepic_classifier.keras"   # adjust if needed
model, CLASS_NAMES, IMG_SIZE, INPUT_NAME, INPUT_DTYPE, LAST_CONV = load_model_and_meta(MODEL_PATH)


Loaded: resnet50_profilepic_classifier.keras


AttributeError: 'str' object has no attribute 'name'

**Reasoning**:
The previous code failed because `INPUT_DTYPE_local` is a string and does not have a `.name` attribute. The print statement needs to be fixed to directly print `INPUT_DTYPE_local`. Also, the global variables need to be updated within the function scope or returned and assigned outside. I will return the values and assign them outside the function.



In [ ]:
# === GradcamInspection: DROP-IN REPLACEMENT for model + metadata loading ===
from pathlib import Path
from keras.models import load_model
import tensorflow as tf
import json

def load_model_and_meta(model_path: str | Path):
    p = Path(model_path)
    if not p.exists():
        raise FileNotFoundError(f"Model file not found: {p.resolve()}")

    # Load the Keras single-file model (no compile needed for inference)
    model = load_model(p, compile=False)

    # Sidecar paths
    classes_path = p.with_suffix(".classes.json")
    meta_path    = p.with_suffix(".meta.json")

    # --- Classes: try sidecar, else infer count and synthesize names ---
    if classes_path.exists():
        try:
            CLASS_NAMES_local = json.loads(classes_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {classes_path.name}: {e}")
            CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]
    else:
        CLASS_NAMES_local = [f"class_{i}" for i in range(model.output_shape[-1])]

    # --- Meta: img_size, last_conv, etc. (optional) ---
    meta_local = {}
    if meta_path.exists():
        try:
            meta_local = json.loads(meta_path.read_text())
        except Exception as e:
            print(f"Warning: couldn't parse {meta_path.name}: {e}")

    # Input info
    inp      = model.inputs[0]
    shape    = inp.shape
    INPUT_NAME_local  = inp.name.split(":")[0]
    INPUT_DTYPE_local = inp.dtype

    # IMG_SIZE: prefer sidecar; else infer from input tensor
    IMG_SIZE_local = int(meta_local.get("img_size") or (shape[1] if shape[1] is not None else 224))

    # Find a valid 4D feature map layer if not provided
    def _find_last_4d(m):
        for lyr in reversed(m.layers):
            try:
                s = getattr(lyr, "output_shape", None)
                if s is None:
                    continue
                if isinstance(s, (list, tuple)) and s and isinstance(s[0], tuple):
                    s = s[0]
                if hasattr(s, "__len__") and len(s) == 4:
                    return lyr.name
            except Exception:
                pass
        raise ValueError("No 4D feature map layer found; cannot run Grad-CAM.")
    LAST_CONV_local = meta_local.get("last_conv") or _find_last_4d(model)

    # Diagnostics
    print(f"Loaded: {p.name}")
    print(f"Input -> name='{INPUT_NAME_local}', dtype={INPUT_DTYPE_local}, shape={tuple(shape)}")
    print(f"IMG_SIZE={IMG_SIZE_local}  |  LAST_CONV='{LAST_CONV_local}'  |  #classes={len(CLASS_NAMES_local)}")

    # NOTE: Preprocessing is assumed to be embedded in the model graph.
    # Do NOT apply external preprocess_input in this notebook.

    return model, CLASS_NAMES_local, IMG_SIZE_local, INPUT_NAME_local, INPUT_DTYPE_local, LAST_CONV_local

# ---- call it ----
MODEL_PATH = "models/resnet50_profilepic_classifier.keras"   # adjust if needed
model, CLASS_NAMES, IMG_SIZE, INPUT_NAME, INPUT_DTYPE, LAST_CONV = load_model_and_meta(MODEL_PATH)

## Build dataset index

Build a DataFrame indexing the images and their labels from the data root directory for the target split.


**Reasoning**:
Implement the logic to build the dataset index from the data root directory, filter it for the target split, handle empty dataframes, define class names if not provided, and create label-to-index and index-to-label mappings as per the instructions.



In [ ]:
def list_images_directory(root: Path, split: str) -> pd.DataFrame:
    print(f"Checking directory: {root / split}")
    rows = []
    base = root / split
    if not base.exists():
        print(f"Directory does not exist: {base}")
        return pd.DataFrame(columns=["path","label","split"])
    for cls_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        for img in cls_dir.rglob("*"):
            if img.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}:
                rows.append({"path": str(img.as_posix()), "label": cls_dir.name, "split": split})
    return pd.DataFrame(rows)

if METADATA_CSV:
    df_all = pd.read_csv(METADATA_CSV)
    need = {"path","label","split"}
    if not need.issubset(set(df_all.columns)):
        raise ValueError(f"CSV must contain columns: {need}")
    df_all["path"] = df_all["path"].astype(str)
else:
    df_train = list_images_directory(DATA_ROOT, "train")
    df_val   = list_images_directory(DATA_ROOT, "val")
    df_test  = list_images_directory(DATA_ROOT, "test")
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)
    print(f"head: {df_all.head()}")
    print(f"shape:{df_all.shape}")

df = df_all[df_all["split"] == TARGET_SPLIT].copy().reset_index(drop=True)

if df.empty:
    raise RuntimeError(f"Failed to build dataset index for split '{TARGET_SPLIT}'. No data found.")

if CLASS_NAMES is None:
    CLASS_NAMES = sorted(df_all["label"].dropna().unique().tolist())

label_to_index = {c:i for i,c in enumerate(CLASS_NAMES)}
index_to_label = {i:c for c,i in label_to_index.items()}

print("Classes:", CLASS_NAMES)
print("Counts:", df["label"].value_counts())

## Data pipeline and preprocessing

Set up a `tf.data` pipeline to load and preprocess images, ensuring the preprocessing matches the model's requirements and handles potential input structure variations (e.g., named inputs).


In [ ]:
def preprocess_image(path: tf.Tensor) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)
    img = tf.cast(img, tf.float32)
    if PREPROCESS.lower() == "resnet50":
        from keras.applications.resnet50 import preprocess_input
        img = preprocess_input(img)
    elif PREPROCESS.lower() == "efficientnet":
        from keras.applications.efficientnet import preprocess_input
        img = efficientnet.preprocess_input(img)
    else:
        img = img / 255.0
    return img

def build_ds(paths: List[str], labels: List[int], batch=BATCH, shuffle=False) -> tf.data.Dataset:
    x = tf.constant(paths, dtype=tf.string)
    y = tf.constant(labels, dtype=tf.int32)
    ds = tf.data.Dataset.from_tensor_slices((x,y))
    if shuffle:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=False)
    ds = ds.map(lambda p,l: (preprocess_image(p), tf.one_hot(l, depth=len(CLASS_NAMES))),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

paths = df["path"].tolist()
labels = [label_to_index[l] for l in df["label"].tolist()]
ds_eval = build_ds(paths, labels, batch=BATCH, shuffle=False)

## Prediction

Run inference on the dataset to get predictions and build a score table including true labels, predicted labels, confidence scores, and correctness.


In [ ]:
probs = []
for xb, yb in ds_eval:
    p = model.predict(xb, verbose=0)
    probs.append(p)
probs = np.vstack(probs)

pred_idx = probs.argmax(axis=1)
pred_lbl = [index_to_label[i] for i in pred_idx]
true_lbl = df["label"].tolist()

conf = probs[np.arange(len(probs)), pred_idx]

score_df = pd.DataFrame({
    "path": paths,
    "true_label": true_lbl,
    "pred_label": pred_lbl,
    "pred_idx": pred_idx,
    "true_idx": [label_to_index[t] for t in true_lbl],
    "confidence": conf
})
score_df["is_correct"] = score_df["true_label"] == score_df["pred_label"]

display(score_df.head())

## Grad-cam utilities

Implement functions for calculating Grad-CAM heatmaps and overlaying them on images, including handling the complexities of accessing intermediate layers and computing gradients with the loaded model. Also include a utility for calculating border attention.


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
from keras import Model

def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # Use the determined CALL_STYLE to call the grad_model
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()

def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

# Remove redundant function definitions
del find_last_conv_layer
del make_gradcam_heatmap
del overlay_heatmap_on_image
del border_attention_fraction

# Redefine the functions with the correct implementations
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # Use the determined CALL_STYLE to call the grad_model
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()

def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

## Visualize grad-cam panels

Select samples (correctly classified and misclassified) for each class and generate/save visualizations showing the original image, heatmap, and overlay with border attention score.


In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple

def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    """Selects correctly classified and misclassified images per class."""
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        # Sort by confidence to get the "most" misclassified/correct
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows

def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())


In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple
from keras import Model # Import Model from keras

def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    """Selects correctly classified and misclassified images per class."""
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        # Sort by confidence to get the "most" misclassified/correct
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows

# Define call_like function again so it's in scope
def call_like(m, x):
    """
    Try calling model with bare tensor, [tensor], and {input_name: tensor}.
    Works around Keras 3 structured-input expectations (e.g., Input(name="image", ...)).
    """
    # 1) Try bare tensor
    try:
        return m(x, training=False)
    except Exception:
        pass
    # 2) Try list-wrapped (single-input models often accept [x])
    try:
        return m([x], training=False)
    except Exception:
        pass
    # 3) Try dict by first input name (assuming INPUT_KEY is defined)
    try:
        if 'INPUT_KEY' in globals() and INPUT_KEY is not None:
             return m({INPUT_KEY: x}, training=False)
    except Exception:
        pass
    # If still failing, raise a clear error
    # Check if x has a shape attribute
    x_shape = getattr(x, 'shape', 'N/A')
    input_names = getattr(m, 'input_names', None)

    raise RuntimeError(
        f"Could not call model with structured inputs. "
        f"Input names={input_names}; got tensor shape={x_shape}"
    )


# Redefine make_gradcam_heatmap to use call_like
def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # <<< KEY CHANGE: call the grad_model using call_like >>>
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


# Redefine overlay_heatmap_on_image
def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

# Redefine border_attention_fraction
def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)

# Redefine find_last_conv_layer
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())

In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple
from keras import Model # Import Model from keras

# Redefine call_like function
def call_like(m, x):
    """
    Try calling model with bare tensor or {input_name: tensor}.
    Works around Keras 3 structured-input expectations.
    """
    # 1) Try bare tensor call (most common and often works)
    try:
        return m(x, training=False)
    except Exception as e_tensor:
        # 2) Try dict by input name (if INPUT_KEY is available)
        if 'INPUT_KEY' in globals() and INPUT_KEY is not None:
            try:
                return m({INPUT_KEY: x}, training=False)
            except Exception as e_dict:
                # 3) Try list-wrapped (less common for single input but worth a shot)
                try:
                    return m([x], training=False)
                except Exception as e_list:
                         # If still failing, raise a clear error
                         x_shape = getattr(x, 'shape', 'N/A')
                         input_names = getattr(m, 'input_names', None)
                         raise RuntimeError(
                             f"Could not call model with structured inputs. "
                             f"Input names={input_names}; got tensor shape={x_shape}. "
                             f"Errors: Tensor call failed ({e_tensor}), Dict call failed ({e_dict}), List call failed ({e_list})"
                         )
        else:
            # INPUT_KEY is not defined or None, report tensor call failure
             x_shape = getattr(x, 'shape', 'N/A')
             input_names = getattr(m, 'input_names', None)
             raise RuntimeError(
                 f"Could not call model with structured inputs. INPUT_KEY is not defined or None. "
                 f"Input names={input_names}; got tensor shape={x_shape}. "
                 f"Error: Tensor call failed ({e_tensor})"
             )
    # This part should ideally not be reached if an exception is raised above
    # But as a fallback, include a generic error if none of the above worked
    x_shape = getattr(x, 'shape', 'N/A')
    input_names = getattr(m, 'input_names', None)
    raise RuntimeError(f"Unexpected error in call_like. Input names={input_names}; got tensor shape={x_shape}.")


# Redefine find_last_conv_layer
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")

# Redefine make_gradcam_heatmap to use call_like
def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    if last_conv_layer_name is None:
        last_conv_layer_name = find_last_conv_layer(model)

    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # <<< KEY CHANGE: call the grad_model using call_like >>>
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


# Redefine overlay_heatmap_on_image
def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

# Redefine border_attention_fraction
def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())

In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
from pathlib import Path
from typing import List, Tuple
from keras import Model # Import Model from keras

# Use the model loading and metadata from cell 'f5550e19'
# This assumes model, CLASS_NAMES, IMG_SIZE, INPUT_KEY, INPUT_DTYPE, LAST_CONV are already defined by cell 'f5550e19' or the model loading code block.

# Redefine preprocess_for_single based on loaded model metadata
def preprocess_for_single(path: str) -> tf.Tensor:
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    # Use IMG_SIZE loaded with the model
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method=tf.image.ResizeMethod.BILINEAR)

    # Use INPUT_DTYPE loaded with the model
    if INPUT_DTYPE == tf.uint8:
        img = tf.clip_by_value(img, 0, 255)
        img = tf.cast(img, tf.uint8)
    else:
        img = tf.cast(img, tf.float32)
        # Assuming preprocessing is handled within the loaded model graph
        # Do NOT apply external preprocess_input here if the model expects raw input

    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    x = tf.expand_dims(img, 0)
    x.set_shape([1, IMG_SIZE, IMG_SIZE, 3])
    return x


# Simplified call_like function
def call_like(m, x):
    """
    Try calling model with bare tensor.
    Fall back to dictionary call if INPUT_KEY is available.
    """
    try:
        # Try plain tensor call first
        return m(x, training=False)
    except Exception as e_tensor:
        # If tensor call fails, try dictionary call using the detected INPUT_KEY
        if 'INPUT_KEY' in globals() and INPUT_KEY is not None:
            try:
                 return m({INPUT_KEY: x}, training=False)
            except Exception as e_dict:
                 # If both fail, raise an informative error
                 x_shape = getattr(x, 'shape', 'N/A')
                 input_names = getattr(m, 'input_names', None)
                 raise RuntimeError(
                     f"Could not call model with structured inputs. "
                     f"Input names={input_names}; got tensor shape={x_shape}. "
                     f"Errors: Tensor call failed ({e_tensor}), Dict call failed ({e_dict})"
                 )
        else:
            # INPUT_KEY not available, report tensor call failure
            x_shape = getattr(x, 'shape', 'N/A')
            input_names = getattr(m, 'input_names', None)
            raise RuntimeError(
                f"Could not call model with structured inputs. INPUT_KEY is not defined or None. "
                f"Input names={input_names}; got tensor shape={x_shape}. "
                f"Error: Tensor call failed ({e_tensor})"
            )


# Redefine find_last_conv_layer
def find_last_conv_layer(model: tf.keras.Model) -> str:
    """Finds the name of the last 4D convolutional layer in the model."""
    # Use the LAST_CONV loaded from metadata if available and valid
    if 'LAST_CONV' in globals() and LAST_CONV is not None:
        try:
            layer = model.get_layer(LAST_CONV)
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Using specified last 4D layer from metadata: {LAST_CONV}")
                return LAST_CONV
        except Exception:
            print(f"Warning: Specified LAST_CONV '{LAST_CONV}' from metadata not found or not 4D. Falling back to auto-detection.")

    # Fallback to auto-detection
    for layer in reversed(model.layers):
        try:
            out_shape = layer.output_shape
            if isinstance(out_shape, list):
                out_shape = out_shape[0]
            if len(out_shape) == 4:
                print(f"Found last 4D layer via auto-detection: {layer.name}")
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4D conv layer found; set TARGET_LAYER_NAME manually.")


# Redefine make_gradcam_heatmap to use call_like and the updated find_last_conv_layer
def make_gradcam_heatmap(
    x: tf.Tensor,
    class_index: int | None = None,
    last_conv_layer_name: str | None = None,
):
    """Generates a Grad-CAM heatmap for a given image tensor."""
    # Determine the last convolutional layer name
    last_conv_layer_name = last_conv_layer_name or find_last_conv_layer(model)


    conv_layer = model.get_layer(last_conv_layer_name)

    # Build a grad model that preserves the original input structure
    grad_model = Model(inputs=model.inputs, outputs=[conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        tape.watch(x)  # make sure gradients can flow wrt the input image
        # <<< KEY CHANGE: call the grad_model using call_like >>>
        conv_outputs, preds = call_like(grad_model, x)

        if class_index is None:
            # Assuming preds is a tensor (batch_size, num_classes) or similar structure
            # If model output is more complex, this might need adjustment
            if isinstance(preds, (list, tuple)):
                 # Assuming the first element is the primary output
                 preds_tensor = preds[0]
            else:
                 preds_tensor = preds

            if preds_tensor.shape.rank == 2:
                class_index = int(tf.argmax(preds_tensor[0]))
            else:
                 # Handle other potential output shapes if necessary
                 raise ValueError(f"Model output shape not supported for auto class index: {preds_tensor.shape}")


        # Calculate the target score
        target = preds_tensor[:, class_index]


    grads = tape.gradient(target, conv_outputs)               # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))            # (C,)
    fmap  = conv_outputs[0]                                   # (H, W, C)
    cam   = tf.reduce_sum(fmap * pooled, axis=-1)             # (H, W)

    cam = tf.maximum(cam, 0.0)
    mx  = tf.reduce_max(cam)
    cam = tf.where(mx > 0, cam / mx, tf.zeros_like(cam))
    return cam.numpy()


# Redefine overlay_heatmap_on_image
def overlay_heatmap_on_image(path: str, heatmap: np.ndarray, alpha=ALPHA, img_size=IMG_SIZE):
    """Overlays a heatmap onto the original image."""
    img = Image.open(path).convert("RGB").resize((img_size, img_size))
    img_np = np.array(img)
    hm = cv2.resize(heatmap, (img_size, img_size))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    # Ensure dimensions match for cv2.addWeighted
    if img_np.shape[-1] == 3 and hm_color.shape[-1] == 3:
         overlay = cv2.addWeighted(hm_color, alpha, img_np[:, :, ::-1], 1.0 - alpha, 0)
    else:
         # Handle cases where image might be grayscale or heatmap is not 3 channels
         # This simplified version assumes both are RGB or compatible
         print("Warning: Image or heatmap not in expected format for overlay. Skipping overlay.")
         return img_np, img_np # Return original image for both

    overlay = overlay[:, :, ::-1] # Convert BGR to RGB
    return img_np, overlay

# Redefine border_attention_fraction
def border_attention_fraction(heatmap: np.ndarray, border_ratio: float = 0.08) -> float:
    """Calculates the fraction of attention in the border region of a heatmap."""
    h, w = heatmap.shape
    b = int(round(min(h, w) * border_ratio))
    core = heatmap[b:h-b, b:w-b].sum() if (b > 0 and (h - 2 * b) > 0 and (w - 2 * b) > 0) else 0.0
    total = heatmap.sum() + 1e-8
    border = total - core
    return float(border / total)


def pick_samples(score_df: pd.DataFrame, per_class_mis:int, per_class_ok:int):
    """Selects correctly classified and misclassified images per class."""
    rows = []
    for cls in CLASS_NAMES:
        sub = score_df[score_df["true_label"] == cls].copy()
        # Sort by confidence to get the "most" misclassified/correct
        mis = sub[~sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_mis)
        ok  = sub[sub["is_correct"]].sort_values("confidence", ascending=False).head(per_class_ok)
        rows.append(("MIS", cls, mis))
        rows.append(("OK",  cls, ok))
    return rows


def panel_for_group(kind: str, cls: str, group_df: pd.DataFrame, save_path: Path):
    """Generates and saves a panel of visualizations for a group of images."""
    n = len(group_df)
    if n == 0:
        print(f"No samples for {kind}: {cls}")
        return

    cols = 3 # Original, Heatmap, Overlay
    rows = math.ceil(n) # Each sample gets one row
    fig_h = max(4, rows * 3) # Adjust figure height based on number of rows
    fig_w = 12

    plt.figure(figsize=(fig_w, fig_h))
    idx = 1
    records = []

    for r in group_df.itertuples(index=False):
        try:
            x = preprocess_for_single(r.path)
            heat = make_gradcam_heatmap(x, class_index=r.pred_idx)
            img_np, overlay = overlay_heatmap_on_image(r.path, heat, alpha=ALPHA, img_size=IMG_SIZE)
            frac = border_attention_fraction(heat)

            # Original Image
            plt.subplot(rows, cols, idx);
            plt.imshow(img_np);
            plt.axis("off");
            plt.title(f"Orig\ntrue={r.true_label}\npred={r.pred_label}\nconf={r.confidence:.2f}")
            idx += 1

            # Heatmap
            plt.subplot(rows, cols, idx);
            plt.imshow(heat, cmap="jet");
            plt.axis("off");
            plt.title("Heatmap")
            idx += 1

            # Overlay
            plt.subplot(rows, cols, idx);
            plt.imshow(overlay);
            plt.axis("off");
            plt.title(f"Overlay\nborder={frac:.2f}")
            idx += 1

            records.append({
                "path": r.path,
                "true_label": r.true_label,
                "pred_label": r.pred_label,
                "confidence": r.confidence,
                "border_attention_frac": frac,
                "is_correct": r.is_correct
            })
        except Exception as e:
            print(f"Error processing image {r.path}: {e}")
            # Add a placeholder or skip the row if processing fails
            # For now, let's just print the error and continue

    if records: # Only try to save if there were successful records
        plt.suptitle(f"{kind}: {cls} — {len(records)} samples", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
        plt.show()

        pd.DataFrame(records).to_csv(save_path.with_suffix(".csv"), index=False)
    else:
        print(f"No successful visualizations for {kind}: {cls}. Skipping save.")
    plt.close() # Close the figure to free memory


# Select samples
samples = pick_samples(score_df, N_MISCLASS_PER_CLASS, N_CORRECT_PER_CLASS)

# Generate and save panels
for kind, cls, gdf in samples:
    slug = f"{kind.lower()}_{cls}".replace(" ", "_")
    out_file = OUT_DIR / f"gradcam_{slug}.png"
    panel_for_group(kind, cls, gdf, out_file)

print("Saved panels to:", OUT_DIR.resolve())

## Visualize grad-cam panels retry 1

Select samples (correctly classified and misclassified) for each class and generate/save visualizations showing the original image, heatmap, and overlay with border attention score.


## Single-image grad-cam helper

Single-image grad-cam helper

#### Instructions
Provide a utility function to generate Grad-CAM for a single specified image.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

def gradcam_single_image(path: str, class_idx: int = None):
    """Generates and displays Grad-CAM visualization for a single image."""
    try:
        # Preprocess the image
        x = preprocess_for_single(path)

        # If class_idx is not provided, get the prediction and determine the class index
        if class_idx is None:
            # Use call_like to handle potential input structure variations
            p = call_like(model, x)
            # Assuming the output is a tensor or list/tuple where the first element is the prediction tensor
            p_tensor = p[0] if isinstance(p, (list, tuple)) else p

            if p_tensor.shape.rank == 2 and p_tensor.shape[0] == 1:
                 class_idx = int(np.argmax(p_tensor[0]))
                 predicted_label = index_to_label.get(class_idx, "Unknown")
                 print(f"Predicted class index: {class_idx} ({predicted_label})")
            else:
                 print(f"Warning: Model output shape {p_tensor.shape} not supported for auto class index determination.")
                 # Fallback: attempt to use argmax on the flattened output if it's a single prediction
                 try:
                     class_idx = int(np.argmax(tf.flatten(p_tensor)))
                     predicted_label = index_to_label.get(class_idx, "Unknown")
                     print(f"Attempted auto class index determination via flatten: {class_idx} ({predicted_label})")
                 except Exception as e_flatten:
                     raise ValueError(f"Could not determine class index automatically. Model output shape: {p_tensor.shape}. Error during flatten attempt: {e_flatten}")

        # Generate the heatmap
        heat = make_gradcam_heatmap(x, class_index=class_idx)

        # Overlay the heatmap on the image
        img_np, overlay = overlay_heatmap_on_image(path, heat, alpha=ALPHA, img_size=IMG_SIZE)

        # Calculate border attention fraction
        frac = border_attention_fraction(heat)

        # Create the visualization panel
        plt.figure(figsize=(10,3))
        plt.subplot(1,3,1); plt.imshow(img_np); plt.axis("off"); plt.title("Original")
        plt.subplot(1,3,2); plt.imshow(heat, cmap="jet"); plt.axis("off"); plt.title("Heatmap")
        plt.subplot(1,3,3); plt.imshow(overlay); plt.axis("off"); plt.title(f"Overlay\nborder={frac:.2f}")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error generating Grad-CAM for image {path}: {e}")
        # Optionally display the original image even if Grad-CAM fails
        try:
            img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            plt.figure(figsize=(4,4))
            plt.imshow(np.array(img))
            plt.title(f"Error: {e}\nOriginal Image")
            plt.axis("off")
            plt.show()
        except Exception as img_e:
            print(f"Also failed to display original image: {img_e}")


# Tools

In [ ]:
def gcs_auth():
  import json
  from google.colab import userdata

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

In [ ]:
def load_zip_data_from_gcs():
  import time

  start_time = time.time()

  bucket_name = 'gs://along-capstone-data'
  source_directory = 'final_zip' # This is the directory in the bucket
  destination_directory = '.' # This is the local destination directory
  zip_file_name = 'final.zip'

  gcs_auth()

  # Copy the directory containing the zip file
  !gsutil -m -q cp -r {bucket_name}/{source_directory} {destination_directory}

  # Construct the local path to the zip file
  local_zip_file_path = f"{destination_directory}/{source_directory}/{zip_file_name}"

  # Unzip the data directory.
  print("Unzipping data...")
  !unzip -o -q {local_zip_file_path} -d {destination_directory}

  end_time = time.time()
  elapsed_time_seconds = end_time - start_time
  elapsed_time_minutes = int(elapsed_time_seconds // 60)
  elapsed_time_remaining_seconds = int(elapsed_time_seconds % 60)


  print(f"Zip download and unzip elapsed time: {elapsed_time_minutes} minutes and {elapsed_time_remaining_seconds} seconds")

# load_zip_data_from_gcs()

In [ ]:
def load_model_from_gcs():
  import time

  start_time = time.time()

  bucket_name = 'gs://along-capstone-data'
  source_directory = 'models' # This is the directory in the bucket
  destination_directory = '.' # This is the local destination directory

  gcs_auth()

  # Copy the directory containing the zip file
  !gsutil -m -q cp -r {bucket_name}/{source_directory} {destination_directory}

  end_time = time.time()
  elapsed_time_seconds = end_time - start_time
  elapsed_time_minutes = int(elapsed_time_seconds // 60)
  elapsed_time_remaining_seconds = int(elapsed_time_seconds % 60)

  print(f"Model download elapsed time: {elapsed_time_minutes} minutes and {elapsed_time_remaining_seconds} seconds")

# load_model_from_gcs()

In [ ]:
def load_model_from_gcs(bucket_name='gs://along-capstone-data', model_name="resnet50_profilepic_classifier.keras", directory='models'):
  """Saves a Keras model to a Google Cloud Storage bucket."""
  import json
  from google.colab import userdata
  from pathlib import Path
  import os

  # Get the service account key from Colab Secrets
  service_account_info = json.loads(userdata.get('GCP_SERVICE_ACCOUNT_KEY'))

  # Define the path to save the service account key file
  key_file_path = 'service_account_key.json'

  # Save the service account key to a file
  with open(key_file_path, 'w') as f:
      json.dump(service_account_info, f)

  # Authenticate gcloud and gsutil using the service account key file
  !gcloud auth activate-service-account --key-file {key_file_path}

  # Define the local path to save the model temporarily
  local_model_dir = Path(directory)
  local_model_dir.mkdir(parents=True, exist_ok=True)
  # local_model_path = local_model_dir / model_name

  # Load the model locally
  # model_to_save.save(local_model_path)
  # print(f"Model saved locally to {local_model_path}")

  # Upload the model to GCS
  gcs_model_path = f"{bucket_name}/{directory}/{model_name}"
  !gsutil cp {gcs_model_path} {local_model_dir}
  print(f"Model loaded to {gcs_model_path}")

  # # Clean up the local model file and directory
  # local_model_path.unlink()
  # local_model_dir.rmdir() # This will only work if the directory is empty after deleting the model file
  # print(f"Local model file {local_model_path} and directory {local_model_dir} removed.")

# load_model_from_gcs()